In [3]:
from pathlib import Path

SRC = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive5")

print("SRC exists:", SRC.exists())

if SRC.exists():
    print("\narchive5 contents:")
    for p in SRC.iterdir():
        print("-", p.name)

    for split in ["train", "val", "valid", "test"]:
        split_dir = SRC / split
        if split_dir.exists():
            print(f"\n=== {split.upper()} ===")
            class_dirs = [p for p in split_dir.iterdir() if p.is_dir()]
            print("Classes:", [p.name for p in class_dirs])

            total = 0
            for class_dir in class_dirs:
                count = len(list(class_dir.glob("*.*")))
                total += count
                print(f"{class_dir.name}: {count}")

            print("Total images:", total)

SRC exists: True

archive5 contents:
- test
- test.csv
- train
- train.csv

=== TRAIN ===
Classes: ['images']
images: 7200
Total images: 7200

=== TEST ===
Classes: ['images']
images: 4800
Total images: 4800


In [4]:
from pathlib import Path
from PIL import Image, ImageEnhance, ImageFilter, ImageFile
import shutil
import numpy as np
import random

ImageFile.LOAD_TRUNCATED_IMAGES = True
random.seed(42)

SRC = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive5")
DST = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive5_aug")

SRC_TRAIN = SRC / "train"

# Some datasets use "val", some use "valid"
if (SRC / "val").exists():
    SRC_VAL = SRC / "val"
elif (SRC / "valid").exists():
    SRC_VAL = SRC / "valid"
else:
    SRC_VAL = None

DST_TRAIN = DST / "train"
DST_VAL = DST / "val"

# Clean old archive5_aug if it already exists
if DST.exists() and DST.name == "archive5_aug":
    shutil.rmtree(DST)

DST_TRAIN.mkdir(parents=True, exist_ok=True)
DST_VAL.mkdir(parents=True, exist_ok=True)

print("Source exists:", SRC.exists())
print("Train exists:", SRC_TRAIN.exists())
print("Val exists:", SRC_VAL.exists() if SRC_VAL else False)
print("Destination ready:", DST.exists())

Source exists: True
Train exists: True
Val exists: False
Destination ready: True


In [5]:
def aug_brightness(img: Image.Image, factor: float) -> Image.Image:
    return ImageEnhance.Brightness(img).enhance(factor)

def aug_contrast(img: Image.Image, factor: float) -> Image.Image:
    return ImageEnhance.Contrast(img).enhance(factor)

def aug_blur(img: Image.Image, radius: float = 1.5) -> Image.Image:
    return img.filter(ImageFilter.GaussianBlur(radius))

def aug_noise(img: Image.Image, noise_level: int = 12) -> Image.Image:
    arr = np.array(img).astype(np.int16)
    noise = np.random.randint(-noise_level, noise_level + 1, arr.shape, dtype=np.int16)
    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

def aug_shadow(img: Image.Image, factor: float = 0.65) -> Image.Image:
    return ImageEnhance.Brightness(img).enhance(factor)

In [6]:
val_copy_count = 0
val_broken_count = 0

if SRC_VAL is None:
    print("No validation folder found.")
else:
    for class_dir in SRC_VAL.iterdir():
        if not class_dir.is_dir():
            continue

        dst_class_dir = DST_VAL / class_dir.name
        dst_class_dir.mkdir(parents=True, exist_ok=True)

        for img_path in class_dir.glob("*.*"):
            try:
                Image.open(img_path).verify()
                shutil.copy2(img_path, dst_class_dir / img_path.name)
                val_copy_count += 1
            except Exception as e:
                print(f"Skipping broken val image: {img_path.name} -> {e}")
                val_broken_count += 1

print("Validation images copied:", val_copy_count)
print("Broken validation images skipped:", val_broken_count)

No validation folder found.
Validation images copied: 0
Broken validation images skipped: 0


In [7]:
train_copy_count = 0
train_broken_count = 0

for class_dir in SRC_TRAIN.iterdir():
    if not class_dir.is_dir():
        continue

    dst_class_dir = DST_TRAIN / class_dir.name
    dst_class_dir.mkdir(parents=True, exist_ok=True)

    for img_path in class_dir.glob("*.*"):
        try:
            Image.open(img_path).verify()
            shutil.copy2(img_path, dst_class_dir / img_path.name)
            train_copy_count += 1
        except Exception as e:
            print(f"Skipping broken train image: {img_path.name} -> {e}")
            train_broken_count += 1

print("Original training images copied:", train_copy_count)
print("Broken training images skipped:", train_broken_count)

Original training images copied: 7200
Broken training images skipped: 0


In [8]:
aug_count = 0
aug_broken_count = 0

for class_dir in SRC_TRAIN.iterdir():
    if not class_dir.is_dir():
        continue

    dst_class_dir = DST_TRAIN / class_dir.name
    dst_class_dir.mkdir(parents=True, exist_ok=True)

    for img_path in class_dir.glob("*.*"):
        stem = img_path.stem
        suffix = img_path.suffix

        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"Skipping broken image during augmentation: {img_path.name} -> {e}")
            aug_broken_count += 1
            continue

        augmentations = [
            ("bright", aug_brightness(img, 1.25)),
            ("dark", aug_brightness(img, 0.75)),
            ("contrast", aug_contrast(img, 1.3)),
            ("blur", aug_blur(img, 1.5)),
            ("noise", aug_noise(img, 12)),
            ("shadow", aug_shadow(img, 0.65)),
        ]

        for tag, aug_img in augmentations:
            new_name = f"{stem}_{tag}{suffix}"
            aug_img.save(dst_class_dir / new_name)
            aug_count += 1

print("Augmented images created:", aug_count)
print("Broken images skipped during augmentation:", aug_broken_count)

Augmented images created: 43200
Broken images skipped during augmentation: 0


In [9]:
final_train_count = 0
final_val_count = 0

print("=== Final Train Counts ===")
for class_dir in sorted(DST_TRAIN.iterdir()):
    if class_dir.is_dir():
        count = len(list(class_dir.glob("*.*")))
        final_train_count += count
        print(f"{class_dir.name}: {count}")

print("\n=== Final Val Counts ===")
for class_dir in sorted(DST_VAL.iterdir()):
    if class_dir.is_dir():
        count = len(list(class_dir.glob("*.*")))
        final_val_count += count
        print(f"{class_dir.name}: {count}")

print("\nFinal train image count:", final_train_count)
print("Final val image count:", final_val_count)

=== Final Train Counts ===
images: 50400

=== Final Val Counts ===

Final train image count: 50400
Final val image count: 0
images: 50400

=== Final Val Counts ===

Final train image count: 50400
Final val image count: 0


In [1]:
import shutil
from pathlib import Path

wrong_aug = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive5_aug")

if wrong_aug.exists():
    shutil.rmtree(wrong_aug)
    print("Deleted wrong archive5_aug")
else:
    print("archive5_aug not found")

archive5_aug not found


In [2]:
from pathlib import Path

SRC = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive5_yolo")

print("SRC exists:", SRC.exists())

if SRC.exists():
    print("\narchive5_yolo contents:")
    for p in SRC.iterdir():
        print("-", p.name)

    for split in ["train", "val", "valid", "test"]:
        split_dir = SRC / split
        if split_dir.exists():
            print(f"\n=== {split.upper()} ===")
            class_dirs = [p for p in split_dir.iterdir() if p.is_dir()]
            print("Classes:", [p.name for p in class_dirs])

            total = 0
            for class_dir in class_dirs:
                count = len(list(class_dir.glob("*.*")))
                total += count
                print(f"{class_dir.name}: {count}")

            print("Total images:", total)

SRC exists: True

archive5_yolo contents:
- train
- train.cache
- val
- val.cache

=== TRAIN ===
Classes: ['1', '2', '3', '4', '5', '6']
1: 137
2: 1879
3: 427
4: 1663
5: 948
6: 706
Total images: 5760

=== VAL ===
Classes: ['1', '2', '3', '4', '5', '6']
1: 34
2: 470
3: 107
4: 416
5: 237
6: 176
Total images: 1440


In [3]:
from pathlib import Path
from PIL import Image, ImageEnhance, ImageFilter, ImageFile
import shutil
import numpy as np
import random

ImageFile.LOAD_TRUNCATED_IMAGES = True
random.seed(42)
np.random.seed(42)

SRC = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive5_yolo")
DST = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive5_yolo_aug")

SRC_TRAIN = SRC / "train"
SRC_VAL = SRC / "val"

DST_TRAIN = DST / "train"
DST_VAL = DST / "val"

# Delete old augmented archive5_yolo_aug if it already exists
if DST.exists():
    shutil.rmtree(DST)
    print("Deleted old archive5_yolo_aug")

DST_TRAIN.mkdir(parents=True, exist_ok=True)
DST_VAL.mkdir(parents=True, exist_ok=True)

print("Source exists:", SRC.exists())
print("Train exists:", SRC_TRAIN.exists())
print("Val exists:", SRC_VAL.exists())
print("Destination ready:", DST.exists())

Source exists: True
Train exists: True
Val exists: True
Destination ready: True


In [4]:
def count_images_by_class(base_dir):
    counts = {}

    for class_dir in sorted(base_dir.iterdir()):
        if class_dir.is_dir():
            images = list(class_dir.glob("*.*"))
            counts[class_dir.name] = len(images)

    return counts

train_counts = count_images_by_class(SRC_TRAIN)
val_counts = count_images_by_class(SRC_VAL)

print("Train counts:")
for cls, count in train_counts.items():
    print(cls, ":", count)

print("\nVal counts:")
for cls, count in val_counts.items():
    print(cls, ":", count)

Train counts:
1 : 137
2 : 1879
3 : 427
4 : 1663
5 : 948
6 : 706

Val counts:
1 : 34
2 : 470
3 : 107
4 : 416
5 : 237
6 : 176


In [5]:
def aug_brightness(img: Image.Image, factor: float) -> Image.Image:
    return ImageEnhance.Brightness(img).enhance(factor)

def aug_contrast(img: Image.Image, factor: float) -> Image.Image:
    return ImageEnhance.Contrast(img).enhance(factor)

def aug_blur(img: Image.Image, radius: float = 1.5) -> Image.Image:
    return img.filter(ImageFilter.GaussianBlur(radius))

def aug_noise(img: Image.Image, noise_level: int = 12) -> Image.Image:
    arr = np.array(img).astype(np.int16)
    noise = np.random.randint(-noise_level, noise_level + 1, arr.shape, dtype=np.int16)
    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

def aug_shadow(img: Image.Image, factor: float = 0.65) -> Image.Image:
    return ImageEnhance.Brightness(img).enhance(factor)

def aug_sharpness(img: Image.Image, factor: float = 1.8) -> Image.Image:
    return ImageEnhance.Sharpness(img).enhance(factor)

def aug_color(img: Image.Image, factor: float = 1.3) -> Image.Image:
    return ImageEnhance.Color(img).enhance(factor)

def aug_gray(img: Image.Image) -> Image.Image:
    return ImageEnhance.Color(img).enhance(0.0)

def aug_flip(img: Image.Image) -> Image.Image:
    return img.transpose(Image.Transpose.FLIP_LEFT_RIGHT)

def aug_rotate_left(img: Image.Image) -> Image.Image:
    return img.rotate(5, expand=False, fillcolor=(0, 0, 0))

def aug_rotate_right(img: Image.Image) -> Image.Image:
    return img.rotate(-5, expand=False, fillcolor=(0, 0, 0))

In [6]:
augment_plan = {
    "1": 12,  # very low class, augment heavily
    "2": 0,   # already high
    "3": 4,   # low class
    "4": 0,   # already high
    "5": 1,   # medium
    "6": 2,   # medium-low
}

print("Augmentation plan:")
for cls, n in augment_plan.items():
    print(f"Class {cls}: create {n} augmented versions per image")

Augmentation plan:
Class 1: create 12 augmented versions per image
Class 2: create 0 augmented versions per image
Class 3: create 4 augmented versions per image
Class 4: create 0 augmented versions per image
Class 5: create 1 augmented versions per image
Class 6: create 2 augmented versions per image


In [7]:
val_copy_count = 0
val_broken_count = 0

for class_dir in sorted(SRC_VAL.iterdir()):
    if not class_dir.is_dir():
        continue

    dst_class_dir = DST_VAL / class_dir.name
    dst_class_dir.mkdir(parents=True, exist_ok=True)

    for img_path in class_dir.glob("*.*"):
        try:
            Image.open(img_path).verify()
            shutil.copy2(img_path, dst_class_dir / img_path.name)
            val_copy_count += 1
        except Exception as e:
            print(f"Skipping broken val image: {img_path.name} -> {e}")
            val_broken_count += 1

print("Validation images copied:", val_copy_count)
print("Broken validation images skipped:", val_broken_count)

Validation images copied: 1440
Broken validation images skipped: 0


In [8]:
train_copy_count = 0
train_broken_count = 0

for class_dir in sorted(SRC_TRAIN.iterdir()):
    if not class_dir.is_dir():
        continue

    dst_class_dir = DST_TRAIN / class_dir.name
    dst_class_dir.mkdir(parents=True, exist_ok=True)

    for img_path in class_dir.glob("*.*"):
        try:
            Image.open(img_path).verify()
            shutil.copy2(img_path, dst_class_dir / img_path.name)
            train_copy_count += 1
        except Exception as e:
            print(f"Skipping broken train image: {img_path.name} -> {e}")
            train_broken_count += 1

print("Original training images copied:", train_copy_count)
print("Broken training images skipped:", train_broken_count)

Original training images copied: 5760
Broken training images skipped: 0


In [9]:
def get_all_augmentations(img):
    return [
        ("bright", aug_brightness(img, 1.25)),
        ("dark", aug_brightness(img, 0.75)),
        ("contrast", aug_contrast(img, 1.3)),
        ("blur", aug_blur(img, 1.5)),
        ("noise", aug_noise(img, 12)),
        ("shadow", aug_shadow(img, 0.65)),
        ("sharp", aug_sharpness(img, 1.8)),
        ("color", aug_color(img, 1.3)),
        ("gray", aug_gray(img)),
        ("flip", aug_flip(img)),
        ("rotleft", aug_rotate_left(img)),
        ("rotright", aug_rotate_right(img)),
    ]

aug_count = 0
aug_broken_count = 0

for class_dir in sorted(SRC_TRAIN.iterdir()):
    if not class_dir.is_dir():
        continue

    class_name = class_dir.name
    num_augments = augment_plan.get(class_name, 0)

    dst_class_dir = DST_TRAIN / class_name
    dst_class_dir.mkdir(parents=True, exist_ok=True)

    print(f"Processing class {class_name} with {num_augments} augmentations per image")

    for img_path in class_dir.glob("*.*"):
        stem = img_path.stem
        suffix = img_path.suffix

        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"Skipping broken image during augmentation: {img_path.name} -> {e}")
            aug_broken_count += 1
            continue

        all_augments = get_all_augmentations(img)
        selected_augments = all_augments[:num_augments]

        for tag, aug_img in selected_augments:
            new_name = f"{stem}_{tag}{suffix}"
            aug_img.save(dst_class_dir / new_name)
            aug_count += 1

print("Augmented images created:", aug_count)
print("Broken images skipped during augmentation:", aug_broken_count)

Processing class 1 with 12 augmentations per image
Processing class 2 with 0 augmentations per image
Processing class 2 with 0 augmentations per image
Processing class 3 with 4 augmentations per image
Processing class 4 with 0 augmentations per image
Processing class 5 with 1 augmentations per image
Processing class 6 with 2 augmentations per image
Augmented images created: 5712
Broken images skipped during augmentation: 0


In [10]:
final_train_count = 0
final_val_count = 0

print("=== Final Train Counts ===")
for class_dir in sorted(DST_TRAIN.iterdir()):
    if class_dir.is_dir():
        count = len(list(class_dir.glob("*.*")))
        final_train_count += count
        print(f"{class_dir.name}: {count}")

print("\n=== Final Val Counts ===")
for class_dir in sorted(DST_VAL.iterdir()):
    if class_dir.is_dir():
        count = len(list(class_dir.glob("*.*")))
        final_val_count += count
        print(f"{class_dir.name}: {count}")

print("\nFinal train image count:", final_train_count)
print("Final val image count:", final_val_count)

=== Final Train Counts ===
1: 1781
2: 1879
3: 2135
4: 1663
5: 1896
6: 2118

=== Final Val Counts ===
1: 34
2: 470
3: 107
4: 416
5: 237
6: 176

Final train image count: 11472
Final val image count: 1440
